In [ ]:
%load_ext autoreload
%autoreload 2
%run ./initialize.ipynb

In [ ]:
import lion_tools
import ipynbname

spark=lion_tools.get_or_create_spark()
display_hub_path=str(ipynbname.path().parent.parent.joinpath("output", "display_hub"))
lion_tools.DisplayHub(display_hub_path).reset()

movies = spark.read.parquet(str(ipynbname.path().parent.parent.joinpath("datasets", "movies.parquet")))


In [ ]:
def display_hub_example(i):
    spark.createDataFrame([[1, "Alice]\\n in hell{}"], [2, "Bob"]], ["id", "name"]).display(
        display=False,
        display_hub_path=display_hub_path,
        title = f"Example Display Hub {i}",
    )

# [display_hub_example(i) for i in range(55)]

In [ ]:
movies.filter('true').display(
    column_filter=True,
    display=False,
    display_hub_path=display_hub_path,
    compact=1,
    title='Compact',
    # paging=True,
    # p=10,
)

In [ ]:
import graphviz

# Extract raw simple physical plan string
raw_plan = movies._jdf.queryExecution().simpleString()
lines = [line.strip() for line in raw_plan.split("\n") if line.strip()]

dot = graphviz.Digraph(comment="Spark Plan", format="svg")
dot.attr("node", shape="box", style="rounded,filled", fillcolor="#eef2ff", fontname="Helvetica")

for i, line in enumerate(lines):
    clean_label = line.lstrip("+- ").split("(")[0]  # Extracts operator name
    dot.node(str(i), clean_label)
    if i > 0:
        dot.edge(str(i), str(i - 1))

dot